# Step 5: Simulate a fly cast

The continuum (finite-element) engine simulates the coupled rod and line
as a bending beam. This notebook runs the two ready-made simulations:

- `simulate_cast`: a generic cast with an idealised rod, line and stroke.
- `simulate_cast1`: Cast #1 from *The Rod & The Cast* (Løvoll & Borger),
  with a rod, line and leader from data files and the stroke fitted to
  the footage.

The Cast #1 match with the footage is qualitative. The stroke is fitted
by eye, and the line is one floppy subdomain without a fly, so it cannot
unroll into a crisp loop.

The `flycastsim` workflow has five steps. Step 1 stands on its own, and
steps 2 to 4 each produce parameters that step 5 uses.

1. Simple 1-D model (`01_simple_1d_model.ipynb`)
2. Rod characteristics (`02_rod_characteristics.ipynb`)
3. Line and leader characteristics (`03_line_and_leader.ipynb`)
4. Casting parameters (`04_casting_parameters.ipynb`)
5. Simulate a fly cast (`05_simulate_cast.ipynb`)

`complete_workflow.ipynb` runs steps 2 to 5 in sequence.

In [ ]:
import numpy as np

from flycastsim import (
    plot_cast_snapshots,
    plot_cast_speeds,
    plot_chord_comparison,
    plot_tip_deflection,
)
from flycastsim.fem import (
    CAST1_ROD_LENGTH,
    cast1_stroke,
    node_index_from_tip,
    node_speed,
    rigid_lever_speed,
    simulate_cast,
    simulate_cast1,
)

## A generic cast

`simulate_cast` returns the time grid `t`, the node positions `X` and `Y`
(one row per time step, one column per node) and the arc-length grid `s`.

In [ ]:
t, X, Y, s = simulate_cast(sweep=np.radians(120), t_stroke=0.4)
print(f"{len(t)} time steps, {X.shape[1]} nodes, {s[-1]:.1f} m of rod and line")
plot_cast_snapshots(t, X, Y)

A shorter stroke with the same sweep moves the tip faster.

In [ ]:
for t_stroke in (0.5, 0.4, 0.3):
    t, X, Y, s = simulate_cast(sweep=np.radians(120), t_stroke=t_stroke)
    tip_speed = node_speed(t, X, Y, X.shape[1] - 1)
    print(f"stroke {t_stroke} s: peak tip speed {tip_speed.max():5.1f} m/s")

## Cast #1

`simulate_cast1` assembles the rig from the bundled component files (see
step 3) and drives it with the Cast #1 stroke (see step 4). Besides the
positions it returns the rod chord length and the tip deflection, which
are the quantities measured from the footage.

In [ ]:
t, X, Y, s, chord, deflection, deflection_vec, rod_tip = simulate_cast1(
    aftm_weight=5)
plot_cast_snapshots(t, X, Y, rod_tip_index=rod_tip)

The chord is the straight-line distance from a point near the butt to the
rod tip. It shortens as the rod bends, and it is compared with the curve
measured from the footage.

In [ ]:
plot_chord_comparison(t, chord)

In [ ]:
plot_tip_deflection(t, deflection)

## Speeds

The rod-tip speed is compared with the speed an imaginary rigid rod of
the same length would give, which shows how much the bending rod adds.

In [ ]:
def cast1_speeds(t, X, Y, s, rod_tip, rod_length, line_distance=0.0):
    """Rod-tip, line and rigid-lever speeds for a Cast #1 run."""
    theta = cast1_stroke()
    butt_angle = np.array([theta(ti) for ti in t])
    line_node = node_index_from_tip(s, line_distance, start=rod_tip,
                                    stop=X.shape[1])
    return (node_speed(t, X, Y, rod_tip),
            node_speed(t, X, Y, line_node),
            rigid_lever_speed(t, X, Y, butt_angle, rod_length))

In [ ]:
tip_speed, line_speed, lever_speed = cast1_speeds(
    t, X, Y, s, rod_tip, CAST1_ROD_LENGTH)
plot_cast_speeds(t, tip_speed, line_speed, lever_speed)

## Line weight and air drag

A heavier line loads the rod more: the tip deflection before RSP, while
the rod is being loaded, grows with the line weight. Air drag can be
switched on with `air_drag=True`.

In [ ]:
for weight, drag in ((4, False), (6, False), (8, False), (6, True)):
    t, X, Y, s, chord, deflection, _, rod_tip = simulate_cast1(
        aftm_weight=weight, air_drag=drag)
    loading = t < 0.0
    print(f"AFTM {weight}, air drag {drag!s:5}: largest tip deflection "
          f"before RSP {np.abs(deflection[loading]).max():.2f} m")

After RSP the simulated rod swings well past straight (the tip
deflection changes sign in the plot above). Treat that rebound with
care: it is part of what makes the match with the footage qualitative.